# Chunking

*Rulebook notebooks: <a href="https://ecoop.github.io/rulebook/rag_at_a_glance.html" target="_parent">RAG at a glance</a> (start here) · Go further: <a href="https://ecoop.github.io/rulebook/embeddings.html" target="_parent">Embeddings</a> · <a href="https://ecoop.github.io/rulebook/evaluation.html" target="_parent">Evaluation</a> · <a href="https://ecoop.github.io/rulebook/reranking.html" target="_parent">Reranking</a> · **Chunking***

<a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Three ways to use this notebook**
> - **Read it, with results:** the <a href="https://ecoop.github.io/rulebook/chunking.html" target="_parent">results page</a> shows it after a fresh run.
> - **Run it in your browser:** <a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/chunking.ipynb" target="_parent">open it in Colab</a> (free; needs a Google account).
>   Already in Colab? Run the cells from the top.
> - **Run it on your machine:** see below.
>
> GitHub's own notebook view shows the code only: no results, and nothing runs there.

Every notebook so far took the passages as given: the poker rules were already cut into the
pieces that retrieval returns and Claude reads. This one goes back to that cut. How a rulebook
is split into chunks decides what retrieval can find, how much Claude has to read, and what an
answer can cite, and it's easy to get subtly wrong.

1. **What a chunk is**, and how Rulebook cuts the rules today.
2. **The naive alternative**: fixed-size windows.
3. **Measuring it fairly**: why the obvious score favours big chunks, and a fairer one.
4. **A bug the gold set found** in Rulebook's own chunker.
5. **An overlooked fix** for the wrong-variant misses from <a href="https://ecoop.github.io/rulebook/evaluation.html" target="_parent">Evaluation</a> and <a href="https://ecoop.github.io/rulebook/reranking.html" target="_parent">Reranking</a>.

This notebook runs **key-free**: the embedder is small and local, and nothing is billed. One
optional cell in section 5 uses the cross-encoder from <a href="https://ecoop.github.io/rulebook/reranking.html" target="_parent">Reranking</a>; without its extra install
(`uv sync --extra rerank-local`, not available on Intel Macs) that cell skips it.

To run it on your own machine, launch from the repo root: `uv run jupyter lab`. On Colab, the
first code cell fetches and installs the repo for you (about a minute).

In [ ]:
# On Colab, fetch the repo first (about a minute). On your own machine this is skipped.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/ecoop/rulebook /content/rulebook
    %pip install -q -e "/content/rulebook[rerank-local]"
    sys.path.insert(0, "/content/rulebook/src")  # this session can't see the new install otherwise

from rulebook.notebook import hit_cell, setup, show_table
setup()

## 1. What a chunk is

A **chunk** is the unit everything downstream works in. Retrieval returns chunks, Claude reads
chunks, and an answer cites a chunk ("poker 3.4"). So the cut matters three ways: a chunk too
small loses its context, a chunk too big buries the answer in text Claude has to read anyway,
and a chunk that doesn't match the rulebook's own structure can't be cited cleanly.

Rulebook cuts at the rulebook's own numbering (`src/rulebook/chunking.py`). It looks for lines
that start like a rule number ("3.4", "II.B.1.", "Rule 5.2") and makes each rule one chunk;
whatever comes before the first rule becomes a **preamble** chunk, and a rule longer than 1,500
characters is split at sentence ends. Here is how the poker rules come out:

In [ ]:
from pathlib import Path

from rulebook.config import settings
from rulebook.eval import domain_chunks, load_gold

chunks = domain_chunks("poker", section_headers=False)
print(f"{len(chunks)} chunks, {min(len(c.text) for c in chunks)} to "
      f"{max(len(c.text) for c in chunks)} characters each")
show_table(
    ["rule", "characters", "text"],
    [[c.rule_id, len(c.text), c.text[:100] + "..."] for c in chunks[:9]],
)

## 2. The naive alternative: fixed-size windows

The simplest chunker ignores structure: cut every N characters, backing up to a space so no
word is split. It works on any text, which is why tutorials start with it. On a rulebook it
cuts wherever the count runs out, so a window can start mid-rule and run on into the next
variant:

In [ ]:
from rulebook.chunk_eval import (
    document_text, fixed_windows, rule_spans, score_cut, section_windows,
)
from rulebook.ingest import extract_pages


def domain_text(domain):
    """The rules' text as the chunker saw it, whitespace collapsed."""
    [doc] = [p for p in (settings.rules_dir / domain).iterdir() if p.suffix == ".md"]
    return document_text([page.text for page in extract_pages(doc)])


text = domain_text("poker")
windows = fixed_windows(text, 400)
straddler = next((a, b) for a, b in windows if "## 3." in text[a:b])
print(f"{len(windows)} windows of up to 400 characters. One of them:\n")
print(text[straddler[0]:straddler[1]])

That window holds the end of five-card stud's rules, the heading, and the start of seven-card
stud's. Retrieved for a seven-card stud question, it hands Claude five-card stud's showdown rule
too, and an answer can't cite it as one rule.

## 3. Measuring it fairly

<a href="https://ecoop.github.io/rulebook/evaluation.html" target="_parent">Evaluation</a>'s scores were made for this kind of question: cut the rules four ways, embed each, and
score each on the gold sets. The obvious way is hit@1 per chunk: is a cited rule *somewhere in*
the first chunk retrieved? The four cuts are Rulebook's rule chunks, fixed windows of 200 and
400 characters, and one chunk per whole section (one per poker variant). Next to hit@1 is how
much text that first chunk is, on average:

In [ ]:
from rulebook.embeddings import LocalEmbedder

embedder = LocalEmbedder(settings.embedding_model)
SETS = ["poker", "checkers", "cornhole"]


def cuts(domain):
    """The four ways of cutting a domain, as (start, end) windows over its text."""
    text = domain_text(domain)
    spans = rule_spans(text, domain_chunks(domain, section_headers=False))
    return text, spans, {
        "rule (Rulebook)": [(s.start, s.end) for s in spans],
        "fixed 200": fixed_windows(text, 200),
        "fixed 400": fixed_windows(text, 400),
        "whole section": section_windows(spans),
    }


scores = {}
for rule_set in SETS:
    text, spans, ways = cuts(rule_set)
    for name, windows in ways.items():
        scores[rule_set, name] = score_cut(
            name, windows, text, spans, load_gold(rule_set), embedder, budget=500)

show_table(
    ["rule set", "cut", "chunks", "hit@1", "characters at rank 1"],
    [[rs, s.name, s.chunks, f"{s.hits_at_1 / s.n:.2f} ({s.hits_at_1}/{s.n})", s.chars_at_1]
     for (rs, _), s in scores.items()],
)

By this score, whole sections win on poker and checkers: on poker, one chunk per variant gets
almost every question right at rank 1. They also hand Claude five times as much text. Push it to the limit and one
chunk holding the whole rulebook scores a perfect 1.0 while telling you nothing. hit@k per
chunk is the right score for one chunker, but it can't compare chunkers, because a bigger chunk
holds more rules and so hits more often.

A fair comparison gives every cut the **same reading budget**: take chunks in retrieval order
until 500 characters are filled (cutting the last one short), the same amount of text for every
cut. Then two questions:

- **hit**: does any of those 500 characters belong to a cited rule?
- **cited share**: what fraction of the 500 characters *is* a cited rule? That's how much of
  what Claude reads is the answer, rather than text around it.

In [ ]:
show_table(
    ["rule set", "cut", "hit within 500 characters", "cited share"],
    [[rs, s.name, f"{s.budget_hits / s.n:.2f} ({s.budget_hits}/{s.n})", f"{s.cited_share:.2f}"]
     for (rs, _), s in scores.items()],
)

At the same budget, whole sections fall to last: their first 500 characters are usually the
start of the right variant, not the rule that answers. Rule chunks find the answer about as
often as fixed windows, and more of what Claude reads is the answer: the highest cited share on
all three rule sets. They also give exact citations, which a window that stops mid-rule can't.

On these clean sample files the gap is modest. On a real rulebook PDF (a table of contents,
headers and footers on every page, rules running across page breaks) fixed windows pick up much
more noise; the budget and cited share are how you'd measure that on your own rules.

The code is in `src/rulebook/chunk_eval.py`; `score_cut` scores any list of windows, so you can
try your own chunker the same way.

## 4. A bug the gold set found

Looking at poker's chunks closely turned up a bug in Rulebook's own chunker. A rule runs until
the next rule starts, and in `poker.md` a section heading sits between the last rule of one
variant and the first of the next:

```
2.6 Showdown. In five-card stud, after the final betting round the best five-card high
hand among the dealt cards wins the pot.

## 3. Seven-Card Stud

3.1 Object. In seven-card stud, ...
```

So the chunker used to end rule 2.6 with "## 3. Seven-Card Stud": five-card stud's showdown rule
carried the name of the *next* variant. The same happened to the last rule of every variant, and
to the preamble. Here is what the chunks looked like, rebuilt from today's chunks by putting back
the text between each chunk and the next, and what it did to the scores:

In [ ]:
from dataclasses import replace

from rulebook.eval import build_scratch_index, evaluate

text = domain_text("poker")
fixed_chunks = domain_chunks("poker", section_headers=False)
spans = rule_spans(text, fixed_chunks)
gaps = [text[s.end:nxt.start].strip() for s, nxt in zip(spans, spans[1:])] + [""]
glued_chunks = [replace(c, text=f"{c.text} {gap}".strip()) for c, gap in zip(fixed_chunks, gaps)]

for c in glued_chunks:
    if c.rule_id == "2.6":
        print("2.6, as the chunker used to cut it:\n" + c.text + "\n")

gold = load_gold("poker")
before = evaluate(gold, k=5, index=build_scratch_index("poker", settings.embedding_model, chunks=glued_chunks))
after = evaluate(gold, k=5, index=build_scratch_index("poker", settings.embedding_model, chunks=fixed_chunks))
show_table(
    ["poker chunks", "hit@1", "hit@3", "MRR"],
    [[name, hit_cell(r, 1), hit_cell(r, 3), f"{r.mrr:.2f}"]
     for name, r in [("heading glued on (the bug)", before), ("heading dropped (fixed)", after)]],
)
for b, f in zip(before.items, after.items, strict=True):
    if b.retrieved_ids[0] != f.retrieved_ids[0]:
        print(f"{b.item.question}  first: {b.retrieved_ids[0]} -> {f.retrieved_ids[0]}"
              f"  (cited: {', '.join(b.item.rule_ids)})")

Small, and invisible in normal use: the answers read fine, and 2.6 still says what it should.
The extra words shift its vector a little, and for two five-card stud questions that was enough
to put 2.6 first. With the fix, the bring-in question gets its rule, 2.3; the deal question now
misses in a different way, which section 5 comes back to. Only the gold set made this visible:
two questions changed, and reading why led back to the chunk text. The fix drops any Markdown
heading from the end of a chunk; it's in `chunking.py` and covered by `tests/test_chunking.py`.

Every chunk in this notebook and in the others is cut with the fix. The numbers in <a href="https://ecoop.github.io/rulebook/evaluation.html" target="_parent">Evaluation</a> and
<a href="https://ecoop.github.io/rulebook/reranking.html" target="_parent">Reranking</a> come from the fixed chunker too.

## 5. An overlooked fix: say which section a rule is in

<a href="https://ecoop.github.io/rulebook/evaluation.html" target="_parent">Evaluation</a> and <a href="https://ecoop.github.io/rulebook/reranking.html" target="_parent">Reranking</a> kept running into one kind of miss: the same rule from the wrong variant.
Here are poker's rank-1 misses again, with the variant each passage comes from:

In [ ]:
VARIANT = {"1": "five-card draw", "2": "five-card stud", "3": "seven-card stud", "4": "Razz", "5": "Omaha"}


def named(rule_id):
    """'4.4' -> '4.4 (Razz)'."""
    variant = VARIANT.get(rule_id.split(".")[0])
    return f"{rule_id} ({variant})" if variant else rule_id


def misses(result):
    rows = []
    for r in result.items:
        if r.retrieved_ids[0] in r.item.rule_ids:
            continue
        rank = next((n for n, rid in enumerate(r.retrieved_ids, start=1) if rid in r.item.rule_ids), None)
        cited_variant = r.item.rule_ids[0].split(".")[0]
        wrong = r.retrieved_ids[0].split(".")[0] not in (cited_variant, "preamble")
        rows.append([r.item.question, " / ".join(map(named, r.item.rule_ids)),
                     named(r.retrieved_ids[0]), f"#{rank}" if rank else "not in top 5",
                     "**wrong variant**" if wrong else ""])
    show_table(["question", "cited", "came first", "cited rule's rank", ""], rows)


misses(after)

Every poker rule already names its variant in its first words ("Seven-card stud has five betting
rounds", "Razz has five betting rounds, ... exactly as in seven-card stud"), and retrieval still
mixes them up. Each rule is embedded on its own, and in a rule that's mostly about betting
rounds, one mention of the variant doesn't weigh much.

A person reading the rulebook never has this problem, because they know which section they're
in. The chunk doesn't: the heading "## 3. Seven-Card Stud" is the one thing that says so, and
it isn't in the rules below it. So put it there. A **section header** (also called a contextual
chunk header) is the section's name at the front of each chunk:

```
Seven-Card Stud: 3.4 Betting rounds. Seven-card stud has five betting rounds, ...
Razz: 4.4 Betting rounds. Razz has five betting rounds, on third through seventh streets, ...
```

It's a one-line change: no model, nothing extra at question time, and the citation is still
"poker 3.4". In Rulebook it's a setting, off by default; here is poker with it on:

In [ ]:
headed = evaluate(gold, k=5, index=build_scratch_index("poker", settings.embedding_model, section_headers=True))
show_table(
    ["poker chunks", "hit@1", "hit@3", "hit@5", "MRR"],
    [[name, hit_cell(r, 1), hit_cell(r, 3), hit_cell(r, 5), f"{r.mrr:.2f}"]
     for name, r in [("rule text only", after), ("with section headers", headed)]],
)
print("Rank-1 misses with section headers:")
misses(headed)

The wrong-variant misses are gone. The seven-card stud betting question now gets 3.4 first, and
the five-card stud deal question gets a five-card stud rule first, though not yet the right one.
What's left is the other kind of miss from <a href="https://ecoop.github.io/rulebook/evaluation.html" target="_parent">Evaluation</a>: a rule from the right variant, just not the
one that answers.

Section headers and reranking fix different things, so they stack. <a href="https://ecoop.github.io/rulebook/reranking.html" target="_parent">Reranking</a>'s rerankers on the
same 20 candidates, with the rule text only and with section headers:

In [ ]:
import time

from rulebook.rerank import CrossEncoderReranker, HybridReranker

rerankers = {"embedding only": None, "hybrid": HybridReranker()}
try:
    rerankers["cross-encoder"] = CrossEncoderReranker()
except ImportError as e:
    print(f"Skipping the cross-encoder here: {e}")

indexes = {
    "rule text only": build_scratch_index("poker", settings.embedding_model, section_headers=False),
    "with section headers": build_scratch_index("poker", settings.embedding_model, section_headers=True),
}
rows = []
for reranker_name, reranker in rerankers.items():
    for chunks_name, index in indexes.items():
        r = evaluate(gold, k=5, index=index, reranker=reranker, candidates=20)
        rows.append([reranker_name, chunks_name, hit_cell(r, 1), hit_cell(r, 3), f"{r.mrr:.2f}"])
show_table(["reranker", "poker chunks", "hit@1", "hit@3", "MRR"], rows)

With hybrid reranking, section headers take poker from 16 to 18 of 22 at rank 1. The
cross-encoder gets 21 of 22 either way: it reads the question and the rule together, so it
already tells the variants apart, and the header adds nothing it didn't have. Where they differ
is cost. A section header is added once, when the index is built; the cross-encoder runs a model
over 20 passages for every question. Without a cross-encoder (on an Intel Mac, say), section
headers plus hybrid is the best setup here.

On checkers and cornhole, section headers change nothing: their rules aren't grouped under
section headings, so there is no section name to add. (A heading used only once, like a
document's title, isn't treated as a section.)

## 6. Turning it on

Section headers are a setting, `CHUNK_SECTION_HEADERS`, off by default. Because it changes the
text that gets embedded, turning it on means rebuilding the index:

```bash
CHUNK_SECTION_HEADERS=true uv run python scripts/build_index.py --domain poker
```

(or set `CHUNK_SECTION_HEADERS=true` in `.env` and rebuild). Today it applies to Markdown rule
files whose numbered rules sit under `##` section headings, like `rules/poker/poker.md`. PDF
rulebooks don't get section headers yet: their section titles ("II. Field") would need to be
recognised first.

## Takeaway

- A **chunk** is what retrieval returns, what Claude reads and what an answer cites. Cutting at
  the rulebook's own rule numbers keeps all three lined up.
- **hit@k per chunk favours big chunks.** To compare chunkers, give each the same reading budget
  and ask how much of it is the answer.
- **Read the chunks, not only the scores.** A heading glued to the wrong rule was invisible in
  the answers; a gold set made it visible, and the chunk text explained it.
- **Give each chunk its context.** A short rule doesn't say which section it's in; a section
  header does, for free, and it removed every wrong-variant miss here.

To check how your own rulebook was cut, build its index and read
`data/index/<domain>/chunks.jsonl`: one chunk per line, with its rule id and text.

**Go further:** more of these notebooks are coming, listed in <a href="https://ecoop.github.io/rulebook/rag_at_a_glance.html" target="_parent">RAG at a glance</a>.